In [ ]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM
 Phase 1: Data Understanding
==================================================================================
 Project   : Customer Retention Intelligence Platform (Final Year Project)
 Author    : Meet Nakrani
 Dataset   : E-Commerce Customer Churn Dataset (Kaggle)
 Files     : E_Comm-Table_1.csv        -> Raw transactional/customer data
             Data_Dict-Table_1.csv     -> Data dictionary (column descriptions)

 Purpose   :
     This script performs the "Data Understanding" phase of the Data Science
     lifecycle (as per the CRISP-DM framework). The goal here is NOT to clean
     or model the data, but to thoroughly explore and document:
         1. Data collection & loading
         2. Data structure (shape, types, memory)
         3. Data dictionary mapping
         4. Missing value analysis
         5. Duplicate record checks
         6. Statistical summary (numerical & categorical)
         7. Target variable (Churn) distribution
         8. Cardinality of categorical features
         9. Initial data quality observations

     Every section prints clearly labeled output so it can be used directly
     as evidence/documentation in the project report.
==================================================================================
"""

import pandas as pd
import numpy as np

# Display settings for cleaner console output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    """Print a formatted section header for readable console output."""
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. DATA COLLECTION
# ----------------------------------------------------------------------------------
section("1. DATA COLLECTION")

DATA_PATH = "E_Comm-Table_1.csv"
DICT_PATH = "Data_Dict-Table_1.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape of dataset -> Rows: {df.shape[0]}, Columns: {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 2. DATA DICTIONARY
# ----------------------------------------------------------------------------------
section("2. DATA DICTIONARY (Column Descriptions)")

data_dict = pd.read_csv(DICT_PATH)
data_dict = data_dict.dropna(axis=1, how="all")          # drop fully empty columns
data_dict.columns = ["Data", "Variable", "Description"]
data_dict = data_dict.dropna(subset=["Variable"]).reset_index(drop=True)
print(data_dict.to_string(index=False))


# ----------------------------------------------------------------------------------
# 3. STRUCTURE OF THE DATASET
# ----------------------------------------------------------------------------------
section("3. DATASET STRUCTURE (First & Last Records)")

print("\n--- First 5 records ---")
print(df.head())

print("\n--- Last 5 records ---")
print(df.tail())

section("3.1 COLUMN NAMES & DATA TYPES")
print(df.dtypes)

section("3.2 MEMORY USAGE & GENERAL INFO")
df.info(memory_usage="deep")


# ----------------------------------------------------------------------------------
# 4. MISSING VALUE ANALYSIS
# ----------------------------------------------------------------------------------
section("4. MISSING VALUE ANALYSIS")

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0]
missing_summary = missing_summary.sort_values("Missing %", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)
    print(f"\nTotal columns with missing values: {missing_summary.shape[0]} out of {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 5. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("5. DUPLICATE RECORD CHECK")

duplicate_rows = df.duplicated().sum()
duplicate_ids = df["CustomerID"].duplicated().sum() if "CustomerID" in df.columns else "N/A"

print(f"Fully duplicated rows        : {duplicate_rows}")
print(f"Duplicated CustomerID values : {duplicate_ids}")


# ----------------------------------------------------------------------------------
# 6. FEATURE TYPE SEGREGATION
# ----------------------------------------------------------------------------------
section("6. FEATURE SEGREGATION (Numerical vs Categorical)")

# CustomerID is an identifier, not a predictive feature -> excluded from analysis lists
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

if "CustomerID" in numerical_cols:
    numerical_cols.remove("CustomerID")

print(f"Numerical columns   ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


# ----------------------------------------------------------------------------------
# 7. STATISTICAL SUMMARY - NUMERICAL FEATURES
# ----------------------------------------------------------------------------------
section("7. STATISTICAL SUMMARY - NUMERICAL FEATURES")
print(df[numerical_cols].describe().T)


# ----------------------------------------------------------------------------------
# 8. STATISTICAL SUMMARY - CATEGORICAL FEATURES
# ----------------------------------------------------------------------------------
section("8. STATISTICAL SUMMARY - CATEGORICAL FEATURES")
print(df[categorical_cols].describe().T)

section("8.1 UNIQUE VALUE COUNTS PER CATEGORICAL FEATURE")
for col in categorical_cols:
    print(f"\n-> {col} ({df[col].nunique()} unique values)")
    print(df[col].value_counts())


# ----------------------------------------------------------------------------------
# 9. TARGET VARIABLE ANALYSIS (Churn)
# ----------------------------------------------------------------------------------
section("9. TARGET VARIABLE ANALYSIS - Churn")

churn_counts = df["Churn"].value_counts()
churn_percent = df["Churn"].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    "Count": churn_counts,
    "Percentage": churn_percent.round(2)
})
print(churn_summary)

imbalance_ratio = round(churn_counts[0] / churn_counts[1], 2)
print(f"\nClass imbalance ratio (Not Churned : Churned) -> {imbalance_ratio} : 1")
print("Observation: The target variable is imbalanced. This must be handled during")
print("the modeling phase (e.g., SMOTE, class weighting, stratified sampling).")


# ----------------------------------------------------------------------------------
# 10. CARDINALITY & CONSTANT FEATURE CHECK
# ----------------------------------------------------------------------------------
section("10. CARDINALITY CHECK (Useful for Identifying IDs / Constant Columns)")

cardinality = df.nunique().sort_values()
print(cardinality)

constant_cols = cardinality[cardinality == 1].index.tolist()
id_like_cols = cardinality[cardinality == len(df)].index.tolist()

print(f"\nConstant columns (no variance): {constant_cols if constant_cols else 'None'}")
print(f"ID-like columns (fully unique): {id_like_cols if id_like_cols else 'None'}")


# ----------------------------------------------------------------------------------
# 11. INITIAL DATA QUALITY OBSERVATIONS (SUMMARY)
# ----------------------------------------------------------------------------------
section("11. INITIAL DATA QUALITY OBSERVATIONS")

observations = f"""
1. Dataset contains {df.shape[0]} customer records and {df.shape[1]} columns.
2. 'CustomerID' is a unique identifier and should be excluded from model training.
3. {missing_summary.shape[0]} columns contain missing values (all numerical),
   ranging from {missing_summary['Missing %'].min()}% to {missing_summary['Missing %'].max()}%
   of records -> requires imputation in the Data Preparation phase.
4. No duplicate rows or duplicate CustomerIDs were found.
5. Target variable 'Churn' is imbalanced ({churn_percent[0].round(1)}% retained vs
   {churn_percent[1].round(1)}% churned) -> must be addressed before model training.
6. Categorical features (e.g., PreferredLoginDevice, PreferredPaymentMode,
   PreferedOrderCat, MaritalStatus, Gender) show inconsistent category naming
   in some cases (e.g., 'Mobile Phone' vs 'Phone', 'CC' vs 'Credit Card') that
   should be standardized during cleaning.
7. No constant or fully-unique (besides CustomerID) columns were detected.
"""
print(observations)

section("DATA UNDERSTANDING PHASE COMPLETE")
print("Next step: Proceed to Data Preparation (missing value treatment, encoding,")
print("category standardization, outlier handling, feature scaling).")